In [ ]:
import os
import json
import scipy.interpolate
import numpy as np
import torch
import torchaudio
import torchaudio.functional as F
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from torchaudio.transforms import Resample, MelSpectrogram, AmplitudeToDB

import logging
from tqdm import tqdm
from faster_whisper import WhisperModel
from transformers import AutoModel, AutoTokenizer,AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

## Load Video & Audio

In [ ]:
#url = "https://download.pytorch.org/torchaudio/tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav"
video_id = "QID80I1IRyI"

path_wsl = f"/mnt/c/Users/dexter.neo/Desktop/LLMVS/{video_id}.mp4"
path_linux = f"/home/dexter/LLaVA-VLS/{video_id}.mp4"

# Check which one actually exists
if os.path.exists(path_wsl):
    url = path_wsl
    print(f"Found file in WSL path: {url}")
elif os.path.exists(path_linux):
    url = path_linux
    print(f"Found file in Linux path: {url}")
    
waveform, sample_rate = torchaudio.load(url)
waveform_mono = waveform[0]
duration_sec = waveform.shape[1] / sample_rate

print(f"Waveform shape: {waveform.shape}")
print(f"Sample rate: {sample_rate} Hz")

In [ ]:
#display(Audio(waveform_mono, rate=sample_rate))

In [ ]:
model_id = "openai/whisper-large-v3-turbo"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize the pipeline
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=model_id,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=device,
    model_kwargs={"attn_implementation": "sdpa"}, # optimizations
)

transcripts = {}

result = asr_pipeline(
        url, 
        chunk_length_s=30, 
        batch_size=8,
        return_timestamps=True
)

In [ ]:
result['chunks']

In [ ]:
try:
    with open(f"/home/dexter/LLaVA-VLS/{video_id}_heatmap.json", 'r') as f:
        data = json.load(f)
except FileNotFoundError:
    with open(f"/mnt/c/Users/dexter.neo/Desktop/LLMVS/{video_id}.mp4", 'r') as f:
        data = json.load(f)

# 3. Prepare Data for Step Plot
times = [entry["start_time"] for entry in data] + [data[-1]["end_time"]]
values = [entry["value"] for entry in data] + [data[-1]["value"]]
plt.plot(times, values)

## Visual Captioning Model (MiniCPM VLM)

In [ ]:
print("Loading Caption Model (MiniCPM-V)...")
caption_model = AutoModel.from_pretrained('/home/dexter/VideoRAG/.checkpoints/MiniCPM-V-2_6-int4', trust_remote_code=True, dtype=torch.bfloat16)
caption_tokenizer = AutoTokenizer.from_pretrained('/home/dexter/VideoRAG/.checkpoints/MiniCPM-V-2_6-int4', trust_remote_code=True)
#from transformers import AutoModel, AutoTokenizer,AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
'''
model_id = "openbmb/MiniCPM-V-2_6-int4"

# Load Model
caption_model = AutoModel.from_pretrained(
    model_id, 
    trust_remote_code=True, 
    dtype=torch.bfloat16
)

# Load Tokenizer
caption_tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True
)
'''

In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer
from decord import VideoReader, cpu  # pip install decord

# 1. Load Model (As you provided)
model_path = '/home/dexter/VideoRAG/.checkpoints/MiniCPM-V-2_6-int4'
model = AutoModel.from_pretrained(model_path, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
model.eval()

vr = VideoReader(video_path, ctx=cpu(0))
fps = vr.get_avg_fps()
step = int(fps)

for i in range(0, len(vr), step):
    frame_data = vr[i]
    
    # Convert to PIL and store
    img = Image.fromarray(frame_data.asnumpy())
    frames.append(img)


print(f"Collected {len(frames)} frames.")

# 4. Chat with the accumulated frames
msgs = [{
    'role': 'user', 
    'content': frames + ["Analyze the gameplay. What is happening?"]
}]

answer = model.chat(
    image=None,
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True,
    temperature=0.7
)
print(answer)

In [ ]:
vr[20].